## Import DuckDB

In [54]:
import duckdb

## Inital Look at the Data

In [55]:

DATA_PATH = '../data/land_registry/pp-*.csv'

In [56]:
duckdb.sql(
    f"""
    SELECT *
    FROM read_csv('{DATA_PATH}', header = false)
    """
)

┌────────────────────────────────────────┬──────────┬─────────────────────┬──────────┬──────────┬──────────┬──────────┬───────────────────┬──────────┬──────────────────┬────────────────┬────────────────┬──────────────────────────────┬───────────────────────┬──────────┬──────────┐
│                column00                │ column01 │      column02       │ column03 │ column04 │ column05 │ column06 │     column07      │ column08 │     column09     │    column10    │    column11    │           column12           │       column13        │ column14 │ column15 │
│                varchar                 │  int64   │      timestamp      │ varchar  │ varchar  │ varchar  │ varchar  │      varchar      │ varchar  │     varchar      │    varchar     │    varchar     │           varchar            │        varchar        │ varchar  │ varchar  │
├────────────────────────────────────────┼──────────┼─────────────────────┼──────────┼──────────┼──────────┼──────────┼───────────────────┼──────────┼───────

## Add Column Names

Unfortunately the CSV file does not include a header row, so we specify these.

[How to access HM Land Registry Price Paid Data](https://www.gov.uk/guidance/about-the-price-paid-data) web site provides useful information to help us do this.  Here is the key information extracted from that site:

In [57]:
land_registry_data = duckdb.sql(
    f"""
    SELECT
        column00 AS id,
        column01 AS price,
        column02 AS date,
        column03 AS postcode,
        column04 AS property_type,
        column05 AS old_new,
        column06 AS duration,
        column07 AS paon,
        column08 AS saon,
        column09 AS street,
        column10 AS locality,
        column11 AS town_city,
        column12 AS district,
        column13 AS county,
        column14 AS ppd_category,
        column15 AS record_type
    FROM read_csv('{DATA_PATH}', header = false)
    """
)

In [58]:
type(land_registry_data)

_duckdb.DuckDBPyRelation

## DuckDBPyRelation?

A DuckDBPyRelation is best thought of as a lazy query (or query plan) rather than the data itself.

- Lazy evaluation
- Composable
- Reusable

LINQ expression in C# or a Spark DataFrame - it's a description of what to compute, not the computed result.

In [88]:
# Displaying forces materialisation:
land_registry_data

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────┬───────────────┬─────────┬──────────┬───────────────────┬─────────┬──────────────────┬────────────────┬────────────────┬──────────────────────────────┬───────────────────────┬──────────────┬─────────────┬───────┐
│                   id                   │ price  │        date         │ postcode │ property_type │ old_new │ duration │       paon        │  saon   │      street      │    locality    │   town_city    │           district           │        county         │ ppd_category │ record_type │ year  │
│                varchar                 │ int64  │      timestamp      │ varchar  │    varchar    │ varchar │ varchar  │      varchar      │ varchar │     varchar      │    varchar     │    varchar     │           varchar            │        varchar        │   varchar    │   varchar   │ int64 │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────┼───────────────┼─────────

## Inspect Data

In [60]:
duckdb.sql("DESCRIBE FROM land_registry_data")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ date          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ property_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ old_new       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ duration      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ paon          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ saon          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ street        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

## Inspect Property Types

In [61]:
land_registry_data = duckdb.sql(
    """
    SELECT * REPLACE (
        CASE property_type
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE property_type
        END AS property_type
    )
    FROM land_registry_data
    """
)

In [62]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

┌───────────────┬─────────┐
│ property_type │  count  │
│    varchar    │  int64  │
├───────────────┼─────────┤
│ Flat          │ 1803230 │
│ Detached      │ 2316342 │
│ Other         │  583366 │
│ Semi-Detached │ 2640880 │
│ Terraced      │ 2713555 │
└───────────────┴─────────┘

In [63]:
land_registry_data = duckdb.sql(
    """
    SELECT * FROM land_registry_data
    WHERE property_type != 'Other'
    """
)

## Add Year Feature

In [64]:
land_registry_data = duckdb.sql(
    """
    SELECT *, year(date) AS year
    FROM land_registry_data
    """
)

## View Data

In [77]:
duckdb.sql(
    """
    SELECT id, date, year, property_type, price
    FROM land_registry_data
    LIMIT 10
    """
)

┌────────────────────────────────────────┬─────────────────────┬───────┬───────────────┬────────┐
│                   id                   │        date         │ year  │ property_type │ price  │
│                varchar                 │      timestamp      │ int64 │    varchar    │ int64  │
├────────────────────────────────────────┼─────────────────────┼───────┼───────────────┼────────┤
│ {3E0330F0-76C8-8D89-E050-A8C062052140} │ 2016-09-21 00:00:00 │  2016 │ Flat          │ 103750 │
│ {3E0330F0-76C9-8D89-E050-A8C062052140} │ 2016-09-16 00:00:00 │  2016 │ Semi-Detached │ 165000 │
│ {3E0330F0-76CA-8D89-E050-A8C062052140} │ 2016-09-02 00:00:00 │  2016 │ Semi-Detached │  90000 │
│ {3E0330F0-76CB-8D89-E050-A8C062052140} │ 2016-08-22 00:00:00 │  2016 │ Terraced      │ 105000 │
│ {3E0330F0-76CC-8D89-E050-A8C062052140} │ 2016-08-30 00:00:00 │  2016 │ Semi-Detached │  67500 │
│ {3E0330F0-76CD-8D89-E050-A8C062052140} │ 2016-07-29 00:00:00 │  2016 │ Semi-Detached │ 127000 │
│ {3E0330F0-76CE-8D8

## Generate Insights

In [81]:
annual_price_by_property_type = duckdb.sql(
    """
    SELECT
        year,
        property_type,
        MEDIAN(price) AS median_price
    FROM land_registry_data
    GROUP BY ALL
    ORDER BY year, property_type
    """
)

In [82]:
annual_price_by_property_type

┌───────┬───────────────┬──────────────┐
│ year  │ property_type │ median_price │
│ int64 │    varchar    │    double    │
├───────┼───────────────┼──────────────┤
│  2016 │ Detached      │     306000.0 │
│  2016 │ Flat          │     200000.0 │
│  2016 │ Semi-Detached │     187500.0 │
│  2016 │ Terraced      │     169000.0 │
│  2017 │ Detached      │     322500.0 │
│  2017 │ Flat          │     210000.0 │
│  2017 │ Semi-Detached │     197000.0 │
│  2017 │ Terraced      │     173000.0 │
│  2018 │ Detached      │     331995.0 │
│  2018 │ Flat          │     210000.0 │
│    ·  │  ·            │         ·    │
│    ·  │  ·            │         ·    │
│    ·  │  ·            │         ·    │
│  2024 │ Semi-Detached │     262500.0 │
│  2024 │ Terraced      │     220000.0 │
│  2025 │ Detached      │     416100.5 │
│  2025 │ Flat          │     230000.0 │
│  2025 │ Semi-Detached │     270000.0 │
│  2025 │ Terraced      │     225000.0 │
│  2026 │ Detached      │     416000.0 │
│  2026 │ Flat  

## Query Plan

In [83]:
annual_price_by_property_type.explain()

''

## Visualise the results

In [84]:
annual_price_by_property_type.pl()

year,property_type,median_price
i64,str,f64
2016,"""Detached""",306000.0
2016,"""Flat""",200000.0
2016,"""Semi-Detached""",187500.0
2016,"""Terraced""",169000.0
2017,"""Detached""",322500.0
…,…,…
2025,"""Terraced""",225000.0
2026,"""Detached""",416000.0
2026,"""Flat""",215000.0


In [85]:
import plotly.express as px

In [89]:
px.line(
    annual_price_by_property_type.pl(),
    x="year",
    y="median_price",
    color="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

## Write to File

In [87]:
from pathlib import Path
Path("../data/price_paid_insights").mkdir(parents=True, exist_ok=True)

duckdb.sql(
    """
    COPY annual_price_by_property_type
    TO '../data/price_paid_insights/annual_price_by_property_type.parquet'
    (FORMAT PARQUET)
    """
)